# Data Preparation for Comprehensive Dashboard
## Backend Processing - Not for Presentation

**Purpose:** Load, clean, join, and prepare all datasets for visualization

**Outputs:**
- Cleaned consumption data
- Joined GCP + Lab data
- Recall data with establishment linkage
- Pre-calculated statistics

**Note:** This notebook contains all data loading code. The dashboard only imports final results.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

DATA_PATH = Path('../data')
OUTPUT_PATH = Path('../data/processed')
OUTPUT_PATH.mkdir(exist_ok=True)

print("✓ Libraries loaded")
print(f"✓ Data path: {DATA_PATH}")
print(f"✓ Output path: {OUTPUT_PATH}")

## 1. Load Consumption Data

In [ ]:
# Load consumption data
consumption_df = pd.read_csv(DATA_PATH / 'usFoodGroupIntakesBySource.csv')

# Filter for 2017-2018, total population
consumption_recent = consumption_df[
    (consumption_df['Survey years:Variable'] == '2017-2018-Mean') & 
    (consumption_df['Demographics'] == 'US consumers aged 2 and above') &
    (consumption_df['Food group'] != 'Energy')
].copy()

# Convert to ounces
def convertToOunces(row):
    if row['Measurement'] == 'Ounces':
        return row['Value']
    elif row['Measurement'] == 'Cups':
        return row['Value'] * 8
    elif row['Measurement'] == 'Grams':
        return row['Value'] / 28.35
    elif row['Measurement'] == 'Teaspoons':
        return row['Value'] * 0.166667
    return 0

consumption_recent['ozEquivalent'] = consumption_recent.apply(convertToOunces, axis=1)

# Categorize
animalKeywords = ['meat', 'poultry', 'eggs', 'seafood', 'fish', 'dairy', 'cured']
plantKeywords = ['vegetable', 'fruit', 'grain', 'legume', 'nut', 'seed', 'soy']

def categorizeFoodSource(foodGroup):
    if pd.isna(foodGroup):
        return 'Other'
    foodLower = str(foodGroup).lower()
    for keyword in animalKeywords:
        if keyword in foodLower:
            return 'Animal'
    for keyword in plantKeywords:
        if keyword in foodLower:
            return 'Plant'
    return 'Other'

consumption_recent['foodType'] = consumption_recent['Food group'].apply(categorizeFoodSource)

# Export
consumption_recent.to_csv(OUTPUT_PATH / 'consumptionCleaned.csv', index=False)
print(f"✓ Consumption data: {len(consumption_recent)} records")

## 2. Load GCP + Lab Data (Joined)

In [ ]:
# Load joined data
joinedData = pd.read_csv(DATA_PATH / 'joinedGcpLabPoultryData.csv')

# Filter for complete data
bothData = joinedData[joinedData['_merge'] == 'both'].copy()

# Export
bothData.to_csv(OUTPUT_PATH / 'gcpLabJoined.csv', index=False)
print(f"✓ GCP+Lab data: {len(bothData)} establishments")

## 3. Load NEW Recall Data (API)

In [ ]:
# Load detailed recall data from API
recallsDf = pd.read_csv(DATA_PATH / 'recallsAllCombined.csv')

# Derive pathogen from summary text
def categorizePathogen(row):
    text = str(row['summary']).lower() + ' ' + str(row['title']).lower()
    if 'listeria' in text:
        return 'Listeria'
    elif 'salmonella' in text:
        return 'Salmonella'
    return 'Unknown'

recallsDf['pathogen'] = recallsDf.apply(categorizePathogen, axis=1)

# Convert dates
recallsDf['recallDate'] = pd.to_datetime(recallsDf['recallDate'], errors='coerce')
recallsDf['closedDate'] = pd.to_datetime(recallsDf['closedDate'], errors='coerce')

# Add derived fields
recallsDf['daysOpen'] = (recallsDf['closedDate'] - recallsDf['recallDate']).dt.days

# Filter for FY2025 alignment (Oct 1, 2024 - Sep 30, 2025)
fy2025_start = pd.to_datetime('2024-10-01')
fy2025_end = pd.to_datetime('2025-09-30')
recallsDf['inFY2025'] = (recallsDf['recallDate'] >= fy2025_start) & (recallsDf['recallDate'] <= fy2025_end)

# Export full dataset
recallsDf.to_csv(OUTPUT_PATH / 'recallsApi.csv', index=False)
print(f"✓ Recall data (API): {len(recallsDf)} recalls")
print(f"  Listeria: {(recallsDf['pathogen'] == 'Listeria').sum()}")
print(f"  Salmonella: {(recallsDf['pathogen'] == 'Salmonella').sum()}")
print(f"  Total pounds: {recallsDf['poundsRecovered'].sum():,.0f}")

# Export FY2025-aligned subset
recallsDf_fy2025 = recallsDf[recallsDf['inFY2025']].copy()
recallsDf_fy2025.to_csv(OUTPUT_PATH / 'recallsApiFY2025.csv', index=False)
print(f"\n✓ Temporal Alignment:")
print(f"  Lab data period: FY2025 (Oct 2024 - Sep 2025)")
print(f"  Recalls in FY2025: {len(recallsDf_fy2025)} of {len(recallsDf)} ({len(recallsDf_fy2025)/len(recallsDf)*100:.1f}%)")
print(f"  Recalls outside period: {len(recallsDf) - len(recallsDf_fy2025)} (context only)")

## 4. Cross-Reference Recalls with Lab Data

In [ ]:
# Create establishment lookup from recalls
recallEstablishments = recallsDf.groupby('establishment').agg({
    'recallNumber': 'count',
    'poundsRecovered': 'sum',
    'relatedToOutbreak': 'sum',
    'pathogen': lambda x: ', '.join(x.unique())
}).reset_index()

recallEstablishments.columns = ['establishment', 'recallCount', 'totalPounds', 'outbreakCount', 'pathogens']

# Try to match with lab data (fuzzy matching needed for production)
# For now, we'll keep them separate and show side-by-side

# Export
recallEstablishments.to_csv(OUTPUT_PATH / 'recallsByEstablishment.csv', index=False)
print(f"✓ Establishments with recalls: {len(recallEstablishments)}")
print(f"\nTop 5 by recall count:")
print(recallEstablishments.nlargest(5, 'recallCount')[['establishment', 'recallCount', 'totalPounds']])

## 5. Calculate Summary Statistics

In [ ]:
# Pre-calculate all statistics for dashboard
stats = {
    # Contamination
    'totalSamples': int(bothData['Lab_TotalSamples'].sum()),
    'totalPositive': int(bothData['Lab_SalmonellaPositive'].sum()),
    'overallRate': float(bothData['Lab_SalmonellaPositive'].sum() / bothData['Lab_TotalSamples'].sum() * 100),
    
    # Welfare
    'totalEstablishments': int(len(bothData)),
    'withMOIs': int((bothData['GCP_TotalMOIs'] > 0).sum()),
    'withNRs': int((bothData['GCP_TotalNRs'] > 0).sum()),
    'moiPercent': float((bothData['GCP_TotalMOIs'] > 0).sum() / len(bothData) * 100),
    
    # Recalls
    'totalRecalls': int(len(recallsDf)),
    'listeriaRecalls': int((recallsDf['pathogen'] == 'Listeria').sum()),
    'salmonellaRecalls': int((recallsDf['pathogen'] == 'Salmonella').sum()),
    'outbreakRecalls': int(recallsDf['relatedToOutbreak'].sum()),
    'totalPoundsRecalled': float(recallsDf['poundsRecovered'].sum()),
    'largestRecall': {
        'establishment': recallsDf.loc[recallsDf['poundsRecovered'].idxmax(), 'establishment'],
        'pounds': float(recallsDf['poundsRecovered'].max()),
        'pathogen': recallsDf.loc[recallsDf['poundsRecovered'].idxmax(), 'pathogen']
    },
    'top5Recalls': recallsDf.nlargest(5, 'poundsRecovered')[['establishment', 'poundsRecovered', 'pathogen', 'relatedToOutbreak']].to_dict('records')
}

# Export as JSON
with open(OUTPUT_PATH / 'dashboardStats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print("✓ Summary statistics calculated")
print(f"\nKey numbers:")
print(f"  Contamination rate: {stats['overallRate']:.2f}%")
print(f"  Welfare concerns: {stats['moiPercent']:.1f}%")
print(f"  Total recalls: {stats['totalRecalls']}")
print(f"  Largest recall: {stats['largestRecall']['establishment']} ({stats['largestRecall']['pounds']:,.0f} lbs)")

## Data Preparation Complete

**Generated Files:**
- `consumptionCleaned.csv` - Consumption data (2017-2018)
- `gcpLabJoined.csv` - GCP + Lab contamination data
- `recallsApi.csv` - Detailed recall data (API)
- `recallsByEstablishment.csv` - Recall summary by company
- `dashboardStats.json` - Pre-calculated statistics

**Next:** Import these into `comprehensiveDashboard.ipynb` for visualization